# Scanpy CPU — FULL Workflow (Mouse Brain 1M)

With regress_out + scale. Subsets to 1M cells to match rsc runs.

**Estimated runtime: 12-20 hours. Run overnight on Libra.**

In [1]:
DATASET = "mouse_brain_1m"

import json, time, os
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import KNeighborsTransformer

sc.settings.verbosity = 0
print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

Scanpy: 1.12


/tmp/ipykernel_1279422/2564081235.py:10: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy: {sc.__version__}")


In [2]:
with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"][DATASET]
prefix = dcfg["pipeline_prefix"]
timings = {}

out = f"write/{prefix}_scanpy_cpu_full"
print(f"Dataset: {dcfg['name']}")
print(f"Output prefix: {out}")

Dataset: Mouse Brain 1.3M
Output prefix: write/mouse_1m_scanpy_cpu_full


In [3]:
# Load — prefer 1M subset if it exists
canonical = dcfg["canonical_h5ad"]
subset_path = canonical.replace(".h5ad", "_1M.h5ad")

t0 = time.time()
if os.path.exists(subset_path):
    adata = sc.read_h5ad(subset_path)
    print(f"Loaded 1M subset: {subset_path}")
else:
    adata = sc.read_h5ad(canonical)
    if adata.n_obs > 1_000_000:
        print(f"Subsetting to 1,000,000 cells (from {adata.n_obs:,})")
        adata = adata[:1_000_000, :].copy()
timings["load"] = time.time() - t0
print(f"Loaded in {timings['load']:.1f}s: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
assert "counts" in adata.layers

Loaded 1M subset: write/mouse_1m_canonical_filtered_1M.h5ad
Loaded in 39.2s: 1,000,000 cells x 22,788 genes


## Normalize + log1p

In [4]:
t0 = time.time()
sc.pp.normalize_total(adata, target_sum=gcfg["target_sum"])
sc.pp.log1p(adata)
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

Normalize + log1p: 8.0s


## HVG

In [5]:
t0 = time.time()
sc.pp.highly_variable_genes(
    adata, layer="counts", n_top_genes=dcfg["n_top_genes"],
    flavor="seurat_v3", subset=False, inplace=True,
)
timings["hvg"] = time.time() - t0
print(f"HVG ({adata.var['highly_variable'].sum()} genes): {timings['hvg']:.1f}s")

HVG (5000 genes): 41.5s


## Subset to HVGs + Regress out + Scale

In [6]:
adata.raw = adata.copy()
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"Subset to HVGs: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

Subset to HVGs: 1,000,000 cells x 5,000 genes


In [7]:
# Auto-detect MT column name
mt_col = None
for candidate in ["pct_counts_mt", "pct_counts_MT", "pct_counts_Mt"]:
    if candidate in adata.obs.columns:
        mt_col = candidate
        break

regress_keys = ["total_counts"]
if mt_col:
    regress_keys.append(mt_col)
print(f"Regress out keys: {regress_keys}")
print("This step is SLOW on CPU at 1M cells. Expect 1-3 hours.")

t0 = time.time()
sc.pp.regress_out(adata, keys=regress_keys)
timings["regress_out"] = time.time() - t0
print(f"Regress out: {timings['regress_out']:.1f}s ({timings['regress_out']/60:.1f} min)")

Regress out keys: ['total_counts', 'pct_counts_mt']
This step is SLOW on CPU at 1M cells. Expect 1-3 hours.
Regress out: 11.3s (0.2 min)


In [8]:
t0 = time.time()
sc.pp.scale(adata, max_value=10)
timings["scale"] = time.time() - t0
print(f"Scale: {timings['scale']:.1f}s")

Scale: 25.4s


## PCA (on scaled data)

In [9]:
t0 = time.time()
sc.pp.pca(
    adata, n_comps=gcfg["pca_n_comps"], svd_solver="arpack",
    random_state=gcfg["random_state"],
)
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

PCA: 76.0s


## Neighbors

In [10]:
t0 = time.time()
transformer = KNeighborsTransformer(
    n_neighbors=dcfg["n_neighbors"], mode="distance",
    metric=gcfg["neighbor_metric"], algorithm="brute",
)
sc.pp.neighbors(
    adata, n_neighbors=dcfg["n_neighbors"], n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca", method=gcfg["neighbor_method"],
    transformer=transformer, metric=gcfg["neighbor_metric"],
    random_state=gcfg["random_state"],
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors: {timings['neighbors']:.1f}s ({timings['neighbors']/60:.1f} min)")

Neighbors: 338.5s (5.6 min)


## Leiden

In [11]:
t0 = time.time()
leiden_kwargs = {
    "resolution": dcfg["leiden_resolution"],
    "flavor": gcfg["scanpy_leiden_flavor"],
    "random_state": gcfg["random_state"],
    "key_added": "leiden",
}
if gcfg.get("scanpy_leiden_n_iterations") is not None:
    leiden_kwargs["n_iterations"] = gcfg["scanpy_leiden_n_iterations"]
sc.tl.leiden(adata, **leiden_kwargs)
timings["leiden"] = time.time() - t0
n_clusters = adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s ({timings['leiden']/60:.1f} min)")

Leiden (32 clusters): 872.8s (14.5 min)


## Checkpoint: save clusters

In [12]:
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)
print(f"Checkpoint: clusters saved ({n_clusters} clusters)")
print(f"Even if DE crashes, clusters are safe.")

Checkpoint: clusters saved (32 clusters)
Even if DE crashes, clusters are safe.


## UMAP

In [13]:
t0 = time.time()
sc.tl.umap(
    adata, min_dist=gcfg["umap_min_dist"], spread=gcfg["umap_spread"],
    init_pos="spectral", random_state=gcfg["random_state"],
)
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s ({timings['umap']/60:.1f} min)")

UMAP: 783.9s (13.1 min)


In [14]:
# Save h5ad before DE (DE takes hours and might crash)
adata.write_h5ad(f"{out}_pre_de.h5ad", compression="gzip")
print("Checkpoint: h5ad saved (no DE yet)")

Checkpoint: h5ad saved (no DE yet)


## Differential Expression

**This is the slowest step. Expect 4-6 hours on CPU at 1M cells.**

In [15]:
t0 = time.time()
sc.tl.rank_genes_groups(
    adata, groupby="leiden", method=gcfg["de_method"],
    corr_method=gcfg["de_corr_method"], use_raw=False, pts=True,
)
timings["de"] = time.time() - t0
print(f"DE: {timings['de']:.1f}s ({timings['de']/3600:.1f} hr)")

DE: 835.2s (0.2 hr)


/home/sjeong7/.local/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:483: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/sjeong7/.local/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:483: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/sjeong7/.local/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:483: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/sjeong7/.local/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:483: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/sjeong7/.local/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:483: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/sjeong7/.local/lib/pytho

## Save all outputs

In [16]:
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(f"{out}_markers.csv", index=False)
markers_filt = markers[(markers["pvals_adj"] < 0.05) & (markers["logfoldchanges"] > 0.1)]
markers_filt.to_csv(f"{out}_markers_filtered.csv", index=False)

pd.DataFrame(
    adata.obsm["X_umap"], index=adata.obs_names.astype(str),
    columns=["UMAP_1", "UMAP_2"],
).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

t0 = time.time()
adata.write_h5ad(f"{out}.h5ad", compression="gzip")
timings["save_h5ad"] = time.time() - t0

spec = {
    "pipeline": "scanpy_cpu_full",
    "dataset": dcfg["name"],
    "workflow": "FULL (with regress_out + scale)",
    "scanpy_version": sc.__version__,
    "regress_out_keys": regress_keys,
    "scale_max_value": 10,
    "n_cells_used": int(adata.n_obs),
    "timings_seconds": timings,
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": dcfg["n_top_genes"],
        "n_clusters": n_clusters,
        "n_marker_rows": len(markers),
        "n_marker_rows_filtered": len(markers_filt),
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    if t > 3600:
        print(f"  {step:20s}: {t:8.1f}s ({t/3600:.1f} hr)")
    elif t > 60:
        print(f"  {step:20s}: {t:8.1f}s ({t/60:.1f} min)")
    else:
        print(f"  {step:20s}: {t:8.1f}s")
total = sum(timings.values())
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/3600:.1f} hr)")
print(f"\nCells: {adata.n_obs:,}")
print(f"Clusters: {n_clusters}")
print(f"Filtered markers: {len(markers_filt)}")


=== Timings ===
  load                :     39.2s
  normalize_log1p     :      8.0s
  hvg                 :     41.5s
  regress_out         :     11.3s
  scale               :     25.4s
  pca                 :     76.0s (1.3 min)
  neighbors           :    338.5s (5.6 min)
  leiden              :    872.8s (14.5 min)
  umap                :    783.9s (13.1 min)
  de                  :    835.2s (13.9 min)
  save_h5ad           :    542.8s (9.0 min)
  TOTAL               :   3574.6s (1.0 hr)

Cells: 1,000,000
Clusters: 32
Filtered markers: 53179


/tmp/ipykernel_1279422/760307873.py:19: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  "scanpy_version": sc.__version__,
